In [19]:
import boto3
import os
import shutil
import warnings
from datetime import date
from dotenv import load_dotenv
from urllib3.exceptions import InsecureRequestWarning
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [20]:
load_dotenv()

True

In [24]:
spark = SparkSession.builder \
    .appName("PySpark_ETL") \
    .config("spark.jars", "../clickhouse-jdbc-0.6.3.jar") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")

In [25]:
jdbc_url = f"jdbc:clickhouse://localhost:{os.getenv('CLICKHOUSE_PORT_1', '8123')}/mart"
properties = {
    "user": "user_mart",
    "password": "strongpassword",
    "driver": "com.clickhouse.jdbc.ClickHouseDriver"}

In [ ]:
# так как фоновые процессы слияния частей данных могут происходить в неопределенное время, 
# и движок ReplacingMergeTree может не успеть удалить дубликаты, 
# мы отфильтруем и возьмем последние загруженные данные
sql_query_purchases = """(
    SELECT purchase_id,
           store_id,
           customer_id,
           product_id,
           total_amount,
           payment_method,
           is_delivery,
           purchase_datetime,
           load_date,
    FROM mart.purchases 
    WHERE load_date = (SELECT max(load_date) FROM mart.purchases)
)"""
df_purchases = spark.read.jdbc(
    url=jdbc_url, 
    table=sql_query_purchases, 
    properties=properties
)

df_purchases.show(5)

+-----------+---------+-----------+----------+------------+--------------+-----------+-------------------+-------------------+
|purchase_id| store_id|customer_id|product_id|total_amount|payment_method|is_delivery|  purchase_datetime|          load_date|
+-----------+---------+-----------+----------+------------+--------------+-----------+-------------------+-------------------+
|  ord-00001|store-017|   cus-1037|  prd-1003|      575.85|          cash|      false|2026-02-03 15:33:38|2026-03-27 19:25:14|
|  ord-00002|store-039|   cus-1027|  prd-1003|      783.45|          card|       true|2026-01-28 15:33:38|2026-03-27 19:25:14|
|  ord-00002|store-039|   cus-1027|  prd-1006|      783.45|          card|       true|2026-01-28 15:33:38|2026-03-27 19:25:14|
|  ord-00002|store-039|   cus-1027|  prd-1012|      783.45|          card|       true|2026-01-28 15:33:38|2026-03-27 19:25:14|
|  ord-00003|store-005|   cus-1027|  prd-1003|     2244.97|          cash|      false|2026-01-23 15:33:38|2026-

In [ ]:
sql_query_products = """(
    SELECT product_id,
           product_name,
           category_name,
           is_organic,
           load_date
    FROM mart.products 
    WHERE load_date = (SELECT max(load_date) FROM mart.products)
)"""
df_products = spark.read.jdbc(
    url=jdbc_url, 
    table=sql_query_products, 
    properties=properties
)
df_products.show(5)

+----------+------------+--------------------+----------+-------------------+
|product_id|product_name|       category_name|is_organic|          load_date|
+----------+------------+--------------------+----------+-------------------+
|  prd-1000|    премьера|мясо, рыба, яйца ...|     false|2026-03-27 19:25:14|
|  prd-1001|  торопливый|мясо, рыба, яйца ...|      true|2026-03-27 19:25:14|
|  prd-1002|     дыхание|      овощи и зелень|      true|2026-03-27 19:25:14|
|  prd-1003|     умолять|   молочные продукты|      true|2026-03-27 19:25:14|
|  prd-1004|  торопливый|мясо, рыба, яйца ...|     false|2026-03-27 19:25:14|
+----------+------------+--------------------+----------+-------------------+
only showing top 5 rows



In [ ]:
sql_query_customers = """(
    SELECT customer_id,
           registration_date,
           is_loyalty_member,
           load_date
    FROM mart.customers 
    WHERE load_date = (SELECT max(load_date) FROM customers)
)"""
df_customers = spark.read.jdbc(
    url=jdbc_url, 
    table=sql_query_customers, 
    properties=properties
)
df_customers.show(5)

+-----------+-------------------+-----------------+-------------------+
|customer_id|  registration_date|is_loyalty_member|          load_date|
+-----------+-------------------+-----------------+-------------------+
|   cus-1000|2026-03-15 15:33:38|             true|2026-03-27 19:25:14|
|   cus-1001|2026-03-15 15:33:38|             true|2026-03-27 19:25:14|
|   cus-1002|2026-03-15 15:33:38|             true|2026-03-27 19:25:14|
|   cus-1003|2026-03-15 15:33:38|             true|2026-03-27 19:25:14|
|   cus-1004|2026-03-15 15:33:38|             true|2026-03-27 19:25:14|
+-----------+-------------------+-----------------+-------------------+
only showing top 5 rows



In [32]:
sql_query_stores = """(
    SELECT store_id, 
           address_id, 
           load_date 
    FROM mart.stores
    WHERE load_date = (SELECT max(load_date) FROM mart.stores)
)"""
df_stores = spark.read.jdbc(
    url=jdbc_url,
    table=sql_query_stores, 
    properties=properties
)
df_stores.show(5)

+---------+--------------------+-------------------+
| store_id|          address_id|          load_date|
+---------+--------------------+-------------------+
|store-001|  482169807853608059|2026-03-27 19:25:14|
|store-002|17691570230506736469|2026-03-27 19:25:14|
|store-003|12846052255788099519|2026-03-27 19:25:14|
|store-004|16251012971226735608|2026-03-27 19:25:14|
|store-005| 4018067191305976778|2026-03-27 19:25:14|
+---------+--------------------+-------------------+
only showing top 5 rows



In [33]:
sql_query_address = """(
    SELECT address_id,
           city,
           load_date
    FROM mart.address
    WHERE load_date = (SELECT max(load_date) FROM mart.address)
)"""
df_address = spark.read.jdbc(
    url=jdbc_url, 
    table=sql_query_address, 
    properties=properties)
df_address.show(5)

+--------------------+-------------+-------------------+
|          address_id|         city|          load_date|
+--------------------+-------------+-------------------+
| 3513008698178146807|    г. калуга|2026-03-27 19:25:14|
|   61540760130738809|    г. калуга|2026-03-27 19:25:14|
|16759602277956548007| г. кетченеры|2026-03-27 19:25:14|
| 5864937174965171697| г. кетченеры|2026-03-27 19:25:14|
|16244484097111366873|г. кисловодск|2026-03-27 19:25:14|
+--------------------+-------------+-------------------+
only showing top 5 rows



In [34]:
# Объединяем все таблицы в одну, берем необходимые колонки и кэшируем
df_final = df_customers.alias("c") \
    .join(df_purchases.alias("pu"), df_customers.customer_id == df_purchases.customer_id , "left") \
    .join(df_products.alias("pr"), df_purchases.product_id == df_products.product_id, "left") \
    .join(df_stores.alias("s"), df_purchases.store_id == df_stores.store_id, "left") \
    .join(df_address.alias("a"), df_stores.address_id == df_address.address_id, "left") \
    .select(
        "pu.purchase_id",
        "pu.store_id",
        "c.customer_id",
        "pu.product_id",
        "pr.category_name",
        "pr.is_organic",
        "total_amount",
        "payment_method",
        "is_delivery",
        "a.city",
        "purchase_datetime",
        F.to_date(F.col("c.registration_date")).alias("registration_date"),
        "c.is_loyalty_member",
        "pu.load_date") \
    .cache() 

df_final.show(5)

+-----------+---------+-----------+----------+--------------------+----------+------------+--------------+-----------+-----------+-------------------+-----------------+-----------------+-------------------+
|purchase_id| store_id|customer_id|product_id|       category_name|is_organic|total_amount|payment_method|is_delivery|       city|  purchase_datetime|registration_date|is_loyalty_member|          load_date|
+-----------+---------+-----------+----------+--------------------+----------+------------+--------------+-----------+-----------+-------------------+-----------------+-----------------+-------------------+
|  ord-00193|store-004|   cus-1007|  prd-1017|мясо, рыба, яйца ...|      true|      499.35|          cash|       true|ст. ребриха|2025-12-18 15:33:38|       2026-03-15|             true|2026-03-27 19:25:14|
|  ord-00193|store-004|   cus-1007|  prd-1016|зерновые и хлебоб...|      true|      499.35|          cash|       true|ст. ребриха|2025-12-18 15:33:38|       2026-03-15|    

In [35]:
df_matrix_features = df_final.groupBy("customer_id") \
    .agg(F.max(
            # Покупал молочные продукты за последние 30 дней
            F.when((F.col("category_name") == "молочные продукты") &
                   (F.col("purchase_datetime") >= F.date_sub(F.current_date(), 30)), True) 
                .otherwise(False)).alias("bought_milk_last_30d"),
        # Покупал фрукты и ягоды за последние 14 дней
        F.max(            
            F.when((F.col("category_name") == "фрукты и ягоды") &
                   (F.col("purchase_datetime") >= F.date_sub(F.current_date(), 14)), True) 
                .otherwise(False)).alias("bought_fruits_last_14d"),
        # Количество покупок овощей и зелени за последние 14 дней
        F.count(
            F.when((F.col("category_name") == "овощи и зелень") &
                   (F.col("purchase_datetime") >= F.date_sub(F.current_date(), 14)), F.col("product_id")) 
                ).alias("count_purchase_vegetables_14"),
        # Количество покупок за последние 30 дней, что бы потом определить делал ли клиент больше 2 покупок за последние 30 дней                        
        F.countDistinct(
            F.when(F.col("purchase_datetime") >= F.date_sub(F.current_date(), 30), 
                                F.col("purchase_id"))).alias("count_purchase_30d"),
        # Количество покупок 
        F.countDistinct(F.col("purchase_id")).alias("count_purchase"),
        # Используется для loyal_customer
        F.max("is_loyalty_member").alias("is_loyalty_member"),
        # Дата последней покупки
        F.max("purchase_datetime").alias("last_purchase"),
        # Покупатель зарегистрировался менее 30 дней назад
        F.max(
            F.when((F.col("registration_date") >= F.date_sub(F.current_date(), 30)), True)
                .otherwise(False)).alias("new_customer"),
        # Пользовался доставкой хотябы раз
        F.max(F.when(F.col("is_delivery") == True, True).otherwise(False)).alias("delivery_user"),
        # Купил хотя бы 1 органический продукт
        F.max(F.when(F.col("is_organic") == True, True).otherwise(False)).alias("organic_preference"),
        # Средняя корзина > 1000 р
        F.when(F.avg("total_amount") > 1000, True).otherwise(False).alias("bulk_buyer"),
        # Средняя корзина < 200 р
        F.when(F.avg("total_amount") < 200, True).otherwise(False).alias("low_cost_buyer"),
        # Покупал хлеб/выпечку хотя бы раз
        F.max(            
            F.when((F.col("category_name") == "зерновые и хлебобулочные изделия"), True)
                .otherwise(False)).alias("buys_bakery"),
        # Количество городов в которых покупатель делает покупки
        F.countDistinct(F.col("city")).alias("count_city"),
        # Покупал мясо/рыбу/яйца за последнюю неделю
        F.max(            
            F.when((F.col("category_name") == "мясо, рыба, яйца и бобовые") &
                   (F.col("purchase_datetime") >= F.date_sub(F.current_date(), 7)), True) 
                .otherwise(False)).alias("bought_meat_last_week"),
        # Делал покупки после 20:00
        F.max(
            F.when((F.hour(F.col("purchase_datetime")) > 20), True).otherwise(False)).alias("night_shopper"),
        # Делал покупки до 10:00
        F.max(
            F.when((F.hour(F.col("purchase_datetime")) < 10), True).otherwise(False)).alias("morning_shopper"),
        # Количество покупок, оплаченные наличными
        F.countDistinct(
            F.when(F.col("payment_method") == "cash", F.col("purchase_id"))).alias("count_purchase_cash"),
        # Количество покупок, оплаченные картой
        F.countDistinct(
            F.when(F.col("payment_method") == "card", F.col("purchase_id"))).alias("count_purchase_card"),
        # Количество покупок в выходной день
        F.countDistinct(
            F.when((F.weekday(F.col("purchase_datetime")) == 5) | (F.weekday(F.col("purchase_datetime")) == 6), 
                   F.col("purchase_id"))).alias("count_purchase_weekend"),
        # Количество категорий в покупках
        F.countDistinct("category_name").alias("count_category_name"),
        # Количество магазинов в которые ходит клиент
        F.countDistinct("store_id").alias("count_store"),
        # Покупка в промежутке между 12 и 15 часами дня
        F.max(
            F.when((F.hour(F.col("purchase_datetime"))).between(12, 14), True).otherwise(False)).alias("early_bird"),
        # Купил на сумму > 2000р за последние 7 дней
        F.max(            
            F.when((F.col("total_amount") >= 2000) &
                   (F.col("purchase_datetime") >= F.date_sub(F.current_date(), 7)), True) 
                .otherwise(False)).alias("recent_high_spender"),
        # Количество покупок с фруктами за последние 30 дней
        F.countDistinct(
            F.when((F.col("purchase_datetime") >= F.date_sub(F.current_date(), 30)) &
                   (F.col("category_name") == "фрукты и ягоды"),
            F.col("category_name"))).alias("count_purchase_fruit_30d"),
        # Количество купленных мясных продуктов за 90 дней
        F.count(
            F.when((F.col("category_name") == "мясо, рыба, яйца и бобовые") & 
                   (F.col("purchase_datetime") >= F.date_sub(F.current_date(), 90)), F.col("product_id")) 
                ).alias("count_meat_90"),
        # Количество покупок с одним товаром в корзине
        # Присоединим к этому датафрейму расчет количества товаров в корзине
    ).join(
        df_final.groupBy("customer_id", "purchase_id") \
            .agg(
                # Количество товаров в корзине
                F.count(F.col("product_id")).alias("count_items")
            ).groupBy("customer_id") \
            .agg(
                # Количество покупок с одним товаром
                F.count(F.when(F.col("count_items") == 1, F.col("count_items"))).alias("count_purchase_items_1"),
                # Среднее количество товаров в корзине
                F.avg(F.col("count_items")).alias("avg_items")
    ),
        "customer_id", "left" 

    # Не покупал овощи и зелень за последние 14 дней
    ).withColumn(
        "not_bought_veggies_14d",
        (F.col("count_purchase_vegetables_14") == 0) 
    # Делал более 2 покупок за последние 30 дней
    ).withColumn(
         "recurrent_buyer", F.col("count_purchase_30d") > 2
    # Не покупал 14-30 дней (ушедший клиент?)
    ).withColumn(
        "inactive_14_30", 
        (F.col("last_purchase") < F.date_sub(F.current_date(), 14)) &
        (F.col("last_purchase") >= F.date_sub(F.current_date(), 30))
    # Лояльный клиент (карта и >= 3 покупки)
    ).withColumn(
        "loyal_customer",
        ((F.col("count_purchase") >= 3) & (F.col("is_loyalty_member") == True))
    # Делал покупки в разных городах
    ).withColumn(
        "multycity_buyer",
        F.col("count_city") >= 2
    # Оплачивал наличными >= 70% покупок
    ).withColumn(
        "prefers_cash",
        (F.col("count_purchase_cash") / F.col("count_purchase") >= 0.7)
    # Оплачивал картой >= 70% покупок
    ).withColumn(
        "prefers_card",
        (F.col("count_purchase_card") / F.col("count_purchase") >= 0.7)
    # Делал >= 60% покупок в выходные
    ).withColumn(
        "weekend_shopper",
        (F.col("count_purchase_weekend") / F.col("count_purchase") >= 0.6)
    # Делал >= 60% покупок в будни
    ).withColumn(
        "weekday_shopper",
        ((F.col("count_purchase") - F.col("count_purchase_weekend")) / F.col("count_purchase") >= 0.6)
    # >= 50% покупок - 1 товар в корзине
    ).withColumn(
        "single_item_buyer",
        (F.col("count_purchase_items_1") / F.col("count_purchase") >= 0.5)
    # Покупал >= 4 разных категорий продуктов
    ).withColumn(
        "varied_shopper",
        (F.col("count_category_name") >= 4)
    # Ходит только в один магазин
    ).withColumn(
        "store_loyal",
        (F.col("count_store") == 1)
    # Ходит в разные магазины
    ).withColumn(
        "switching_loyal",
        (F.col("count_store") > 1)
    # Среднее количество позиций в корзине >= 4
    ).withColumn(
        "family_shopper",
        (F.col("avg_items") >= 4)
    # Не совершал ни одной покупки (только регистрация)
    ).withColumn(
        "no_purchases",
        (F.col("count_purchase") == 0)
    # >= 3 покупок фруктов за 30 дней
    ).withColumn(
        "fruit_lover",
        (F.col("count_purchase_fruit_30d") >= 3)
    # Не купил ни одного мясного продукта за 90 дней
    ).withColumn(
        "vegetarian_profile",
        (F.col("count_meat_90") == 0)
    )

In [36]:
df_res = df_matrix_features.select(
    "customer_id", 
    "bought_milk_last_30d", # Покупал молочные продукты за последние 30 дней
    "bought_fruits_last_14d", # Покупал фрукты и ягоды за последние 14 дней
    "not_bought_veggies_14d", # Не покупал овощи и зелень за последние 14 дней
    "recurrent_buyer", # Делал более 2 покупок за последние 30 дней
    "inactive_14_30", # Не покупал 14-30 дней (ушедший клиент?)
    "new_customer", # Покупатель зарегистрировался менее 30 дней назад
    "delivery_user", # Пользовался доставкой хотябы раз
    "organic_preference", # Купил хотя бы 1 органический продукт
    "bulk_buyer", # Средняя корзина > 1000 р
    "low_cost_buyer", # Средняя корзина < 200 р
    "buys_bakery", # Покупал хлеб/выпечку хотя бы раз
    "loyal_customer", # Лояльный клиент (карта и >= 3 покупки)
    "multycity_buyer", # Делал покупки в разных городах
    "bought_meat_last_week", # Покупал мясо/рыбу/яйца за последнюю неделю
    "night_shopper", # Делал покупки после 20:00
    "morning_shopper", # Делал покупки до 10:00,
    "prefers_cash", # Оплачивал наличными >= 70% покупок
    "prefers_card", # Оплачивал картой >= 70% покупок
    "weekend_shopper", # Делал >= 60% покупок в выходные
    "weekday_shopper", # Делал >= 60% покупок в будни
    "single_item_buyer",  # >= 50% покупок - 1 товар в корзине
    "varied_shopper", # Покупал >= 4 разных категорий продуктов
    "store_loyal", # Ходит только в один магазин
    "switching_loyal", # Ходит в разные магазины
    "family_shopper", # Среднее количество позиций в корзине >= 4
    "early_bird", # Покупка в промежутке между 12 и 15 часами дня
    "no_purchases", # Не совершал ни одной покупки (только регистрация)
    "recent_high_spender", # Купил на сумму > 2000р за последние 7 дней 
    "fruit_lover", # >= 3 покупок фруктов за 30 дней
    "vegetarian_profile" # Не купил ни одного мясного продукта за 90 дней
).fillna(False).orderBy("customer_id")
df_res.show(10)

+-----------+--------------------+----------------------+----------------------+---------------+--------------+------------+-------------+------------------+----------+--------------+-----------+--------------+---------------+---------------------+-------------+---------------+------------+------------+---------------+---------------+-----------------+--------------+-----------+---------------+--------------+----------+------------+-------------------+-----------+------------------+
|customer_id|bought_milk_last_30d|bought_fruits_last_14d|not_bought_veggies_14d|recurrent_buyer|inactive_14_30|new_customer|delivery_user|organic_preference|bulk_buyer|low_cost_buyer|buys_bakery|loyal_customer|multycity_buyer|bought_meat_last_week|night_shopper|morning_shopper|prefers_cash|prefers_card|weekend_shopper|weekday_shopper|single_item_buyer|varied_shopper|store_loyal|switching_loyal|family_shopper|early_bird|no_purchases|recent_high_spender|fruit_lover|vegetarian_profile|
+-----------+-----------

In [37]:
current_data = date.today().strftime('%Y_%m_%d')
path_data_result = Path().resolve().parent / 'data/result' 
file_name =  f"analytic_result_{current_data}"

df_res.coalesce(1).write.format("csv").mode("overwrite").option("header", "true").save(str(path_data_result / file_name))


In [38]:
try:
    for file in (path_data_result / file_name).glob("part-*.csv"):
        file.rename(path_data_result / (file_name + '.csv'))
    shutil.rmtree(path_data_result / file_name)
except Exception as e:
    print(e)

In [39]:
with warnings.catch_warnings():
    warnings.filterwarnings('ignore', category=InsecureRequestWarning)

    session = boto3.Session(
            aws_access_key_id=os.getenv("access_key"),
            aws_secret_access_key=os.getenv("secret"),
        )
        
    s3_client = session.client('s3', endpoint_url='https://s3.ru-7.storage.selcloud.ru', verify=False)
    bucket_name = 'data-engineer-s3-practice'
    s3_client.upload_file(str(path_data_result / (file_name + '.csv')),
                          bucket_name,
                          ('analitic_for_pikcha_marts' + '/' + str(file_name + '.csv')))

In [40]:
df_final.unpersist()
spark.stop()